In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm
!pip install catboost


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "/kaggle/input/q3-ka-ai-2026/Q3_data.csv")

df = pd.read_csv(csv_path)



In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
def check_missing_values(df):

  # Get missing values using pandas
  missing_values = df.isnull().sum()

  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])

  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):

  #TODO: get duplicated data using pandas
  duplicates = df.duplicated().sum()

  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
# Task 4: Write your code here: Apply feature scaling to numerical features (Use StandardScaler)


from sklearn.preprocessing import StandardScaler

target_col = "Target"

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns
numerical_cols = numerical_cols.drop(target_col)   # <-- THIS LINE IS THE FIX

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])




In [ ]:
# Task 5: Write your code here:
target_col = df.columns[-1]
counts = df[target_col].value_counts(normalize=True)
print(counts)
print("Imbalanced" if counts.max() > 0.6 else "Not imbalanced")


In [ ]:


X = df.drop("Target", axis=1)
y = df["Target"]



In [ ]:
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer

# Fix NaNs first
imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(X)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = []

for train_idx, test_idx in kf.split(X_imputed):
    X_train, X_test = X_imputed[train_idx], X_imputed[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    scores.append(accuracy_score(y_test, y_pred))

print("CV Accuracy:", sum(scores) / len(scores))


In [ ]:
# Task 1: Write your code here:
theta = model.coef_[0]

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': np.abs(theta)
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 16))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

golden_feature = feature_importance.iloc[0]['feature']
golden_importance = feature_importance.iloc[0]['importance']

print("Golden Feature:", golden_feature)
print("Importance:", golden_importance)


In [ ]:
# Task Bonus: Write your code here: